# Does the model see more than it says?

At each layer of Qwen3-VL-2B we ask: could a simple read-out tell "stripes on the left" from
"stripes on the right" from that layer's numbers alone? And at what contrast? Then we compare
with the model's own answer, from the same images.

- **2 frequencies**: 4 c/deg (where humans are best) and 16 c/deg
- **11 contrasts**, 0.0001% to 10%, half a log unit apart: much fainter than in
  `run_model.ipynb`, because the model has no internal noise and a read-out can use any change
- **100 pairs** per point (the same Gabor left, then right), 4,400 images

Results go to `../data/layer_readout.csv`. Run after `run_model.ipynb` (one GPU at a time).

## Setup

In [1]:
import os
import sys

inColab = "google.colab" in sys.modules

if inColab:
  from google.colab import drive
  drive.mount("/content/drive")
  os.chdir("/content/drive/MyDrive/Projects/vlm-psychophysics/chapters/contrast_sensitivity/code")
  os.system("pip install -q -U transformers")

sys.path.insert(0, os.getcwd())
print("running in Colab:", inColab)

Mounted at /content/drive
running in Colab: True


In [2]:
import time
import torch
import numpy  as np
import pandas as pd
from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline        import make_pipeline
from sklearn.preprocessing   import StandardScaler
from transformers       import AutoModelForImageTextToText, AutoProcessor
from transformers.utils import logging

import stimuli

logging.disable_progress_bar()
logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float32
modelName = "Qwen/Qwen3-VL-2B-Instruct"
model = AutoModelForImageTextToText.from_pretrained(modelName, dtype=dtype, device_map=device).eval()
proc  = AutoProcessor.from_pretrained(modelName)

question = "Where is the striped pattern: left or right? Answer with one word."
leftIds  = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ["left", "Left"]]
rightIds = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ["right", "Right"]]


def chatInputs(question, img=None):
  content = []
  if img is not None:
    content.append({"type": "image", "image": stimuli.to_image(img)})
  content.append({"type": "text", "text": question})
  return proc.apply_chat_template([{"role": "user", "content": content}], tokenize=True, add_generation_prompt=True,
                                  return_dict=True, return_tensors="pt").to(device)


blank      = np.full((stimuli.CANVAS_PX[1], stimuli.CANVAS_PX[0]), float(stimuli.MEAN_GREY))
baseInputs = chatInputs(question, blank)
print("model loaded")

model loaded


## Where does the lean towards "left" come from?

The model's preference for "left" (its score for "left" over "right") with no image at all, with
a plain grey image, and with the two words swapped in the question.

In [3]:
def lean(inputs):
  with torch.no_grad():
    logp = torch.log_softmax(model(**inputs, logits_to_keep=1).logits[0, -1].float(), 0)
  return float(torch.logsumexp(logp[leftIds], 0) - torch.logsumexp(logp[rightIds], 0))

swapped = "Where is the striped pattern: right or left? Answer with one word."
print("no image:                   %.2f" % lean(chatInputs(question)))
print("plain grey image:           %.2f" % lean(baseInputs))
print("grey image, words swapped:  %.2f" % lean(chatInputs(swapped, blank)))
print("(above 0: prefers left)")

no image:                   1.40
plain grey image:           1.18
grey image, words swapped:  -0.31
(above 0: prefers left)


## The numbers at each layer

For every image we keep, at each layer:

- **vision layers 1-24 and the merger**: the average of the left half's squares minus the
  average of the right half's (where the stripes are is all the read-out needs)
- **language-model layers 0-28**: the numbers at the place where the answer is written

and the model's answer score, as in `run_model.ipynb`.

In [4]:
vision = model.model.visual
seen   = {}

def keep(name):
  def hook(module, args, out):
    seen[name] = (out[0] if isinstance(out, tuple) else out).detach()
  return hook

hooks  = [b.register_forward_hook(keep("vision %d" % (i + 1))) for i, b in enumerate(vision.blocks)]
hooks += [vision.merger.register_forward_hook(keep("merger"))]

P = stimuli.PATCH
nRows, nCols = stimuli.CANVAS_PX[1] // P, stimuli.CANVAS_PX[0] // P
# the column of each square, in the order the model gets them (2 x 2 blocks, one after another)
squareCol = np.array([bw * 2 + mw for bh in range(nRows // 2) for bw in range(nCols // 2) for mh in range(2) for mw in range(2)])
tokenCol  = np.arange(nRows // 2 * nCols // 2) % (nCols // 2)


def leftMinusRight(x, cols, half):
  return (x[:, cols < half].mean(1) - x[:, cols >= half].mean(1)).cpu().numpy().astype(np.float32)


def layerNumbers(greys):
  n = len(greys)
  inputs = {k: v.repeat(n, *[1] * (v.dim() - 1)) for k, v in baseInputs.items() if k != "pixel_values"}
  inputs["pixel_values"] = torch.from_numpy(np.concatenate([stimuli.pixel_values(g) for g in greys])).to(device, dtype)
  with torch.no_grad():
    out = model(**inputs, output_hidden_states=True, logits_to_keep=1)
  layers = {}
  for i in range(len(vision.blocks)):
    name = "vision %d" % (i + 1)
    layers[name] = leftMinusRight(seen[name].reshape(n, len(squareCol), -1), squareCol, nCols // 2)
  layers["merger"] = leftMinusRight(seen["merger"].reshape(n, len(tokenCol), -1), tokenCol, nCols // 4)
  for i, h in enumerate(out.hidden_states):
    layers["language %d" % i] = h[:, -1].float().cpu().numpy()
  logp = torch.log_softmax(out.logits[:, -1].float(), 1)
  answer = (torch.logsumexp(logp[:, leftIds], 1) - torch.logsumexp(logp[:, rightIds], 1)).cpu().numpy()
  return layers, answer

layers, answer = layerNumbers([blank])
print(len(layers), "layers kept;", "answer score for grey: %.2f" % answer[0])

54 layers kept; answer score for grey: 1.18


## The read-out

For one frequency, contrast and layer: 100 pairs, 200 images. A logistic regression learns
"left or right" from 80 pairs and is tested on the other 20, five times over (each pair is
tested once). It is scored like the model: a pair is right if the read-out says "left" more
strongly for the left display. Guessing gives 50%.

In [5]:
def readout(X, side, pair):
  correct = np.zeros(pair.max() + 1)
  for train, test in GroupKFold(n_splits=5).split(X, side, pair):
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000))
    clf.fit(X[train], side[train])
    score = clf.decision_function(X[test])
    for p in np.unique(pair[test]):
      left  = score[(pair[test] == p) & (side[test] == 1)][0]
      right = score[(pair[test] == p) & (side[test] == 0)][0]
      correct[p] = 1.0 if left > right else 0.5 if left == right else 0.0
  return correct.mean()

## The run

In [6]:
resultsFile = "../data/layer_readout.csv"
frequencies = [4, 16]
contrasts   = [round(float(c), 8) for c in np.logspace(-6, -1, 11)]
pairsPerPoint, batchPairs = 100, 8
rng = np.random.default_rng(1)

rows = []
startTime = time.time()
for f in frequencies:
  for c in contrasts:
    phases = rng.uniform(0, 2 * np.pi, pairsPerPoint)
    numbers, answers = {}, []
    for b in range(0, pairsPerPoint, batchPairs):
      greys = [stimuli.draw_trial(dict(frequency=f, contrast=c, phase=ph), s)
               for ph in phases[b:b + batchPairs] for s in ("left", "right")]
      layers, answer = layerNumbers(greys)
      for name, x in layers.items():
        numbers.setdefault(name, []).append(x)
      answers.append(answer)
    side = np.tile([1, 0], pairsPerPoint)                 # 1 = Gabor on the left
    pair = np.repeat(np.arange(pairsPerPoint), 2)
    answers = np.concatenate(answers)
    said = np.mean(np.where(answers[0::2] > answers[1::2], 1.0, np.where(answers[0::2] == answers[1::2], 0.5, 0.0)))
    rows.append(dict(frequency=f, contrast=c, layer="answer", order=99, correct=said))
    for k, (name, xs) in enumerate(numbers.items()):
      rows.append(dict(frequency=f, contrast=c, layer=name, order=k, correct=readout(np.concatenate(xs), side, pair)))
    pd.DataFrame(rows).to_csv(resultsFile, index=False)
    print("%g c/deg, contrast %.6f%%: answer %.2f, best layer %.2f  (%.0f s)"
          % (f, 100 * c, said, max(r["correct"] for r in rows[-len(numbers):]), time.time() - startTime))
print("done")

4 c/deg, contrast 0.000100%: answer 0.47, best layer 1.00  (62 s)
4 c/deg, contrast 0.000316%: answer 0.43, best layer 1.00  (128 s)
4 c/deg, contrast 0.001000%: answer 0.63, best layer 1.00  (196 s)
4 c/deg, contrast 0.003162%: answer 0.42, best layer 1.00  (262 s)
4 c/deg, contrast 0.010000%: answer 0.43, best layer 1.00  (325 s)
4 c/deg, contrast 0.031623%: answer 0.41, best layer 1.00  (386 s)
4 c/deg, contrast 0.100000%: answer 0.29, best layer 1.00  (446 s)
4 c/deg, contrast 0.316228%: answer 0.96, best layer 1.00  (505 s)
4 c/deg, contrast 1.000000%: answer 1.00, best layer 1.00  (563 s)
4 c/deg, contrast 3.162278%: answer 1.00, best layer 1.00  (621 s)
4 c/deg, contrast 10.000000%: answer 1.00, best layer 1.00  (679 s)
16 c/deg, contrast 0.000100%: answer 0.53, best layer 1.00  (742 s)
16 c/deg, contrast 0.000316%: answer 0.47, best layer 1.00  (806 s)
16 c/deg, contrast 0.001000%: answer 0.49, best layer 1.00  (872 s)
16 c/deg, contrast 0.003162%: answer 0.56, best layer 1.00 

In [7]:
table = pd.DataFrame(rows).pivot_table(index=["frequency", "order", "layer"], columns="contrast", values="correct")
print(table.round(2).to_string())

contrast                     0.000001  0.000003  0.000010  0.000032  0.000100  0.000316  0.001000  0.003162  0.010000  0.031623  0.100000
frequency order layer                                                                                                                    
4         0     vision 1         1.00      1.00      1.00      1.00      1.00      1.00      1.00      1.00       1.0       1.0       1.0
          1     vision 2         1.00      1.00      1.00      1.00      1.00      1.00      1.00      1.00       1.0       1.0       1.0
          2     vision 3         1.00      1.00      1.00      1.00      1.00      1.00      1.00      1.00       1.0       1.0       1.0
          3     vision 4         1.00      1.00      1.00      1.00      1.00      1.00      1.00      1.00       1.0       1.0       1.0
          4     vision 5         1.00      1.00      1.00      1.00      1.00      1.00      1.00      1.00       1.0       1.0       1.0
          5     vision 6         1